Environment

In [16]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [17]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [18]:
import os
import json
import pycomet
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import LinearMEGraphNet

Plotting

In [19]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [20]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [21]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [22]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [23]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Construct graph

In [24]:
# Adjacency matrix
kf, kb = 0.0, 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
    kb = kb + reaction.bwd.k.numpy().squeeze()[0]
k = kf + kb.T

In [55]:
# Dimension
nb_pts, nb_nodes = nm.shape
# Nodes
node_attr = nm[0].reshape(1,nb_nodes,1)
node_attr = np.tile(node_attr, (nb_pts,1,1))
# Edges
k_idx = np.where(k > 0)
edge_attr = (k[k_idx] * na[0]).reshape(1,-1,1)
edge_attr = np.tile(edge_attr, (nb_pts,1,1))
edge_index = np.array(k_idx)
# Graph
graph = [node_attr, edge_attr]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)

Load linear master equation graph model

In [26]:
gnet = LinearMEGraphNet(static_edge_index=edge_index)
call = lambda graph: gnet(graph, training=False) + graph[0]

In [27]:
shapes = tf.nest.map_structure(lambda x: tuple(x.shape[1:]), graph)
inputs = [tf.keras.Input(shape) for shape in shapes]
model = tf.keras.Model(inputs, call(inputs))

In [49]:
def add_noise(x, stddev=0.02):
    rnd = tf.random.get_global_generator()
    dx = rnd.normal(
        shape=tf.shape(x),
        mean=0.0,
        stddev=stddev,
        dtype=x.dtype,
    )
    return x * (1 + dx)

In [57]:
graphn = tf.nest.map_structure(tf.identity, graph)
# graphn[0] = add_noise(graphn[0], stddev=0.02)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[1])
  y = model(graphn)
  print(y.shape)
dy_dx = g.batch_jacobian(y, graphn[1])
dy_dx = tf.squeeze(dy_dx)
dy_dx[0]

(940, 3, 1)


<tf.Tensor: shape=(940, 3, 6), dtype=float64, numpy=
array([[[-1.3761644e+23, -1.3761644e+23,  1.4485941e+09,  0.0000000e+00,
          1.4485941e+09,  0.0000000e+00],
        [ 1.3761644e+23,  0.0000000e+00, -1.4485941e+09, -1.4485941e+09,
          0.0000000e+00,  1.4485941e+09],
        [ 0.0000000e+00,  1.3761644e+23,  0.0000000e+00,  1.4485941e+09,
         -1.4485941e+09, -1.4485941e+09]],

       [[-1.3761644e+23, -1.3761644e+23,  1.4485941e+09,  0.0000000e+00,
          1.4485941e+09,  0.0000000e+00],
        [ 1.3761644e+23,  0.0000000e+00, -1.4485941e+09, -1.4485941e+09,
          0.0000000e+00,  1.4485941e+09],
        [ 0.0000000e+00,  1.3761644e+23,  0.0000000e+00,  1.4485941e+09,
         -1.4485941e+09, -1.4485941e+09]],

       [[-1.3761644e+23, -1.3761644e+23,  1.4485941e+09,  0.0000000e+00,
          1.4485941e+09,  0.0000000e+00],
        [ 1.3761644e+23,  0.0000000e+00, -1.4485941e+09, -1.4485941e+09,
          0.0000000e+00,  1.4485941e+09],
        [ 0.0000000e+00

In [61]:
print(tf.reduce_sum(dy_dx[0], axis=0))

tf.Tensor([-2.7523288e+23  1.3761644e+23  1.3761644e+23], shape=(3,), dtype=float64)


In [ ]:
dnmdt_pred = model([node_attr, edge_attr]) / const.UNA
dnmdt_pred = dnmdt_pred.numpy().squeeze()

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=dnmdt_pred[1:],
    xref=t.squeeze()[1:],
    yref=dnmdt[1:],
    labels=[r'$t$ [s]', r'$\omega$ [m$^{-3}$s$^{-1}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
    figname=path_to_figs + "/source"
)